## Neighbourhood simulation
Where does a sound-driven signal actually go in the real wiring, when the path is not picked in advance?

---

### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc
import duckdb
import networkx as nx

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file) # directly to pandas dataframe
traced = df[df['status'] == 'Traced']
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy handle on the big file, nothing is loaded until a filtered read asks for it
dataset = ds.dataset(connections_file, format="feather")


# common functions and lookups copied from the earlier sections of 05

# get outgoing connections of specified neurons
def query_edges(neuron_ids, min_weight=3):
    if not isinstance(neuron_ids, (list, set, tuple)):
        neuron_ids = [neuron_ids]
    expr = pc.field("body_pre").isin(neuron_ids) & (pc.field("weight") >= min_weight)
    return dataset.to_table(filter=expr).to_pandas()

def grow_neighbourhood(start_ids, n_hops, min_weight=3):
    if not isinstance(start_ids, (list, set, tuple)):
        start_ids = [start_ids]

    seen = set(start_ids)       # every neuron reached so far, starting ones included
    to_check = set(start_ids)   # neurons whose outgoing connections we look up this hop
    all_edges = []

    for hop in range(1, n_hops + 1):
        hop_edges = query_edges(to_check, min_weight=min_weight)
        all_edges.append(hop_edges)

        to_check = set(hop_edges['body_post']) - seen   # targets not seen before
        # seen |= to_check
        seen.update(to_check)

        # prints each hop statistics
        print(f"hop {hop}: {len(hop_edges)} edges, {len(to_check)} new neurons, {len(seen)} total")
        if not to_check:
            break

    return pd.concat(all_edges, ignore_index=True), seen

def run_lif(edges, external_input, pulse_ticks, threshold, decay_factor, time_steps, verbose=True):
    # who each neuron sends charge to: {neuron: [(target, weight), ...]}
    outgoing = {}
    for pre, post, weight in edges[['body_pre', 'body_post', 'weight']].itertuples(index=False):
        outgoing.setdefault(pre, []).append((post, weight))

    charge = {}   # only neurons currently holding charge; a missing neuron means 0
    events = []   # (tick, neuron) for every firing

    for t in range(time_steps):
        if t in pulse_ticks:
            if verbose:
                print(f"tick {t}: external input added to {len(external_input)} neuron(s)")
            for nid, amount in external_input.items():
                charge[nid] = charge.get(nid, 0) + amount

        # who fires this tick, judged on charge before anything resets
        fired = [nid for nid, c in charge.items() if c >= threshold]

        # leak for everyone
        for nid in charge:
            charge[nid] *= decay_factor

        # firing neurons pass their weights on, then reset
        for nid in fired:
            events.append((t, nid))
            for post, weight in outgoing.get(nid, []):
                charge[post] = charge.get(post, 0) + weight
        for nid in fired:
            charge.pop(nid, None)
            if verbose and nid in motor_info:
                info = motor_info[nid]
                print(f"tick {t}: vnc_motor {nid} fired ({info['somaNeuromere']}, {info['instance']})")

    return events # returns a set of tick, id

motor_info = (traced[traced['superclass'] == 'vnc_motor']
              .set_index('bodyId')[['somaNeuromere', 'instance']]
              .fillna({'somaNeuromere': 'unknown'}) # added missing values
              .to_dict('index'))

# regrow neighbourhood again
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]

jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]

jo_ids = set(jo_auditory['bodyId'])

edges_jo, seen_jo = grow_neighbourhood(jo_ids, n_hops=4, min_weight=10)

sources = {nid: 10_000 for nid in jo_ids}

##### Changing the decay

**goal:** check whether the sharp change found in the tick section comes from the wiring or from the decay setting. the model's decay is a number picked by hand (0.8), so if the change only appeared at 0.8, it would be a property of the model and not of the fly. this section moves the decay and finds where the sharp change lands each time.

the cliff at 361/362 was found at decay 0.8 only. the decay is how much charge leaks away each tick, so if the cliff came from the leak specifically, changing it should move the cliff a lot. if it came from the wiring, it should barely move.

scanning every threshold for every decay value would be slow, so use a binary search: start with a threshold known to flood (50) and one known not to (2500), test the middle, and keep whichever half still contains the switch from flood to not flood. each test halves the gap, so around 12 runs find the exact threshold.

- "flood" means more than 1,000 neurons fired. the two regimes seen so far were about 35,000 and about 160, so any cutoff between them gives the same answer
- assumption: raising the threshold never brings flooding back, which matched the scan above
- decay 0.8 is included as a check, it should give 361 again

In [33]:
def find_cliff(decay, lo=50, hi=2500, flood_size=1000):
    # assumes lo floods and hi does not; narrows until they are 1 apart
    while hi - lo > 1:
        mid = (lo + hi) // 2
        ev = run_lif(edges_jo, sources, pulse_ticks={0}, threshold=mid,
                     decay_factor=decay, time_steps=80, verbose=False)
        n_fired = len({nid for t, nid in ev})
        if n_fired > flood_size:
            lo = mid
        else:
            hi = mid
    return lo   # highest threshold that still floods

for decay in [0.8, 0.7, 0.9]:
    print(f"decay {decay}: last flooding threshold = {find_cliff(decay)}")

decay 0.8: last flooding threshold = 361
decay 0.7: last flooding threshold = 353
decay 0.9: last flooding threshold = 362


> - decay 0.8: 361 (same as the scan, so the search is set up correctly)
> - decay 0.7: 353
> - decay 0.9: 362
>
> the cliff moves, but only a little. less leak (0.9) raises it by 1 and more leak (0.7) lowers it by 8.
>
> this matches the trace of 10739: after a neuron fires its charge resets to 0, so the charge at tick 4 is mostly a fresh sum of weights from the previous tick, with little left over for the decay to act on. so the position of the cliff suggests that it is set mainly by the wiring (about 350 to 362), and only the exact integer depends on the decay.
>
> what holds across all three: there is a cliff, flooding below it, and nothing above it.
>
> not checked: whether the abdomen leads at decay 0.7 and 0.9, and the cutoff and input size were not varied.